In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
#download video
!gdown "https://drive.google.com/uc?id=11RAtkdmDtjWd9ZEuFlksrS48m2UbHyS5"

Downloading...
From: https://drive.google.com/uc?id=11RAtkdmDtjWd9ZEuFlksrS48m2UbHyS5
To: /content/video8.mp4
100% 7.61M/7.61M [00:00<00:00, 49.3MB/s]


**USING YOLO11n**

In [ ]:
from ultralytics import YOLO
from collections import defaultdict
import numpy as np
import cv2
import csv
import time

# Load YOLO model (Change this when testing other YOLO variants)
model = YOLO("yolo11n.pt")  # You can replace with "yolo11s.pt", "yolo11l.pt", etc.

# Open video
cap = cv2.VideoCapture("/content/video8.mp4")

# Get video properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_input = cap.get(cv2.CAP_PROP_FPS)

# Output video writer
out = cv2.VideoWriter("output_person_tracked.avi",
                      cv2.VideoWriter_fourcc(*'XVID'),
                      fps_input,
                      (frame_width, frame_height))

# For storing tracking trail of each ID
track_history = defaultdict(lambda: [])

# For FPS calculation
prev_time = 0
fps_list = []  # To store FPS values per frame

# For accuracy
total_detected = 0  # Total people detected
frame_count = 0     # Total frames processed

# Create a set to store unique person IDs
unique_ids = set()

# Loop through video
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Perform tracking on the current frame (only for person class: class 0)
    results = model.track(source=frame, persist=True, classes=[0])

    # Only process if there are tracked objects
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        annotated_frame = results[0].plot()

        # Update total detections and frame count
        total_detected += len(track_ids)  # Count of persons detected in this frame
        frame_count += 1

        # Draw tracking trails
        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            unique_ids.add(track_id)
            track = track_history[track_id]
            track.append((float(x), float(y)))
            if len(track) > 30:
                track.pop(0)

            # Draw tracking lines as polylines
            points = np.hstack(track).astype(np.int32).reshape(-1, 1, 2)
            cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 0, 0), thickness=10)

        # Calculate FPS
        cur_time = time.time()
        fps = 1 / (cur_time - prev_time)
        prev_time = cur_time
        fps_list.append(fps)

        # Display FPS on video
        cv2.putText(annotated_frame, "fps: " + str(int(fps)), (10, 70),
                    cv2.FONT_HERSHEY_PLAIN, 3, (255, 0, 255), 3)

        # Save output frame
        out.write(annotated_frame)

# Release everything
cap.release()
out.release()
cv2.destroyAllWindows()

# Save tracking trail summary to CSV
with open("tracking_summary.csv", mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["person_id", "tracking_trail_points", "start_point", "end_point"])
    for person_id, trail in track_history.items():
        if len(trail) >= 2:
            start_point = trail[0]
            end_point = trail[-1]
            writer.writerow([person_id, str(trail), str(start_point), str(end_point)])

# ----------- RESULTS SUMMARY -----------

#number of persons detected
total_detected = len(unique_ids)

# Calculate average FPS
if len(fps_list) > 0:
    average_fps = sum(fps_list) / len(fps_list)
else:
    average_fps = 0

# Calculate average detections per frame (as a simple accuracy measure)
if frame_count > 0:
    avg_detections_per_frame = total_detected / frame_count
else:
    avg_detections_per_frame = 0


# Display results
print("\n==== YOLO Model Evaluation Summary ====")
print(f"Model: yolo11n.pt")
print(f"Total frames processed: {frame_count}")
print(f"Total persons detected: {total_detected}")
print(f"Average detections per frame: {avg_detections_per_frame:.2f}")
print(f"Average FPS: {average_fps:.2f}")
print("=======================================\n")
print("Video saved as 'output_person_tracked.avi'")
print("Tracking summary saved to 'tracking_summary.csv'")



0: 384x640 17 persons, 11.7ms
Speed: 2.5ms preprocess, 11.7ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 persons, 15.6ms
Speed: 4.4ms preprocess, 15.6ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 persons, 13.4ms
Speed: 3.3ms preprocess, 13.4ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 18 persons, 12.0ms
Speed: 3.2ms preprocess, 12.0ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 persons, 10.2ms
Speed: 3.5ms preprocess, 10.2ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 persons, 11.2ms
Speed: 3.3ms preprocess, 11.2ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 persons, 13.3ms
Speed: 3.3ms preprocess, 13.3ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 persons, 12.1ms
Speed: 4.1ms preprocess, 12.1ms inference, 1.7ms postprocess per image at

**USING YOLO11s**

In [ ]:
from ultralytics import YOLO
from collections import defaultdict
import numpy as np
import cv2
import csv
import time

# Load YOLO model (Change this when testing other YOLO variants)
model = YOLO("yolo11s.pt")  # You can replace with "yolo11s.pt", "yolo11l.pt", etc.

# Open video
cap = cv2.VideoCapture("/content/video8.mp4")

# Get video properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_input = cap.get(cv2.CAP_PROP_FPS)

# Output video writer
out = cv2.VideoWriter("output_person_tracked.avi",
                      cv2.VideoWriter_fourcc(*'XVID'),
                      fps_input,
                      (frame_width, frame_height))

# For storing tracking trail of each ID
track_history = defaultdict(lambda: [])

# For FPS calculation
prev_time = 0
fps_list = []  # To store FPS values per frame

# For accuracy
total_detected = 0  # Total people detected
frame_count = 0     # Total frames processed

# Create a set to store unique person IDs
unique_ids = set()

# Loop through video
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Perform tracking on the current frame (only for person class: class 0)
    results = model.track(source=frame, persist=True, classes=[0])

    # Only process if there are tracked objects
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        annotated_frame = results[0].plot()

        # Update total detections and frame count
        total_detected += len(track_ids)  # Count of persons detected in this frame
        frame_count += 1

        # Draw tracking trails
        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            unique_ids.add(track_id)
            track = track_history[track_id]
            track.append((float(x), float(y)))
            if len(track) > 30:
                track.pop(0)

            # Draw tracking lines as polylines
            points = np.hstack(track).astype(np.int32).reshape(-1, 1, 2)
            cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 0, 0), thickness=10)

        # Calculate FPS
        cur_time = time.time()
        fps = 1 / (cur_time - prev_time)
        prev_time = cur_time
        fps_list.append(fps)

        # Display FPS on video
        cv2.putText(annotated_frame, "fps: " + str(int(fps)), (10, 70),
                    cv2.FONT_HERSHEY_PLAIN, 3, (255, 0, 255), 3)

        # Save output frame
        out.write(annotated_frame)

# Release everything
cap.release()
out.release()
cv2.destroyAllWindows()

# Save tracking trail summary to CSV
with open("tracking_summary.csv", mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["person_id", "tracking_trail_points", "start_point", "end_point"])
    for person_id, trail in track_history.items():
        if len(trail) >= 2:
            start_point = trail[0]
            end_point = trail[-1]
            writer.writerow([person_id, str(trail), str(start_point), str(end_point)])

# ----------- RESULTS SUMMARY -----------

#number of persons detected
total_detected = len(unique_ids)

# Calculate average FPS
if len(fps_list) > 0:
    average_fps = sum(fps_list) / len(fps_list)
else:
    average_fps = 0

# Calculate average detections per frame (as a simple accuracy measure)
if frame_count > 0:
    avg_detections_per_frame = total_detected / frame_count
else:
    avg_detections_per_frame = 0


# Display results
print("\n==== YOLO Model Evaluation Summary ====")
print(f"Model: yolo11s.pt")
print(f"Total frames processed: {frame_count}")
print(f"Total persons detected: {total_detected}")
print(f"Average detections per frame: {avg_detections_per_frame:.2f}")
print(f"Average FPS: {average_fps:.2f}")
print("=======================================\n")
print("Video saved as 'output_person_tracked.avi'")
print("Tracking summary saved to 'tracking_summary.csv'")



0: 384x640 32 persons, 12.8ms
Speed: 2.5ms preprocess, 12.8ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 32 persons, 14.7ms
Speed: 3.2ms preprocess, 14.7ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 13.4ms
Speed: 3.2ms preprocess, 13.4ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 14.2ms
Speed: 4.4ms preprocess, 14.2ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 11.1ms
Speed: 3.3ms preprocess, 11.1ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 34 persons, 12.7ms
Speed: 3.3ms preprocess, 12.7ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 33 persons, 14.4ms
Speed: 3.1ms preprocess, 14.4ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 34 persons, 12.7ms
Speed: 3.3ms preprocess, 12.7ms inference, 1.7ms postprocess per image at

**USING YOLO11l**

In [ ]:
from ultralytics import YOLO
from collections import defaultdict
import numpy as np
import cv2
import csv
import time

# Load YOLO model (Change this when testing other YOLO variants)
model = YOLO("yolo11l.pt")  # You can replace with "yolo11s.pt", "yolo11l.pt", etc.

# Open video
cap = cv2.VideoCapture("/content/video8.mp4")

# Get video properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_input = cap.get(cv2.CAP_PROP_FPS)

# Output video writer
out = cv2.VideoWriter("output_person_tracked.avi",
                      cv2.VideoWriter_fourcc(*'XVID'),
                      fps_input,
                      (frame_width, frame_height))

# For storing tracking trail of each ID
track_history = defaultdict(lambda: [])

# For FPS calculation
prev_time = 0
fps_list = []  # To store FPS values per frame

# For accuracy
total_detected = 0  # Total people detected
frame_count = 0     # Total frames processed

# Create a set to store unique person IDs
unique_ids = set()

# Loop through video
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Perform tracking on the current frame (only for person class: class 0)
    results = model.track(source=frame, persist=True, classes=[0])

    # Only process if there are tracked objects
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        annotated_frame = results[0].plot()

        # Update total detections and frame count
        total_detected += len(track_ids)  # Count of persons detected in this frame
        frame_count += 1

        # Draw tracking trails
        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            unique_ids.add(track_id)
            track = track_history[track_id]
            track.append((float(x), float(y)))
            if len(track) > 30:
                track.pop(0)

            # Draw tracking lines as polylines
            points = np.hstack(track).astype(np.int32).reshape(-1, 1, 2)
            cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 0, 0), thickness=10)

        # Calculate FPS
        cur_time = time.time()
        fps = 1 / (cur_time - prev_time)
        prev_time = cur_time
        fps_list.append(fps)

        # Display FPS on video
        cv2.putText(annotated_frame, "fps: " + str(int(fps)), (10, 70),
                    cv2.FONT_HERSHEY_PLAIN, 3, (255, 0, 255), 3)

        # Save output frame
        out.write(annotated_frame)

# Release everything
cap.release()
out.release()
cv2.destroyAllWindows()

# Save tracking trail summary to CSV
with open("tracking_summary.csv", mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["person_id", "tracking_trail_points", "start_point", "end_point"])
    for person_id, trail in track_history.items():
        if len(trail) >= 2:
            start_point = trail[0]
            end_point = trail[-1]
            writer.writerow([person_id, str(trail), str(start_point), str(end_point)])

# ----------- RESULTS SUMMARY -----------

#number of persons detected
total_detected = len(unique_ids)

# Calculate average FPS
if len(fps_list) > 0:
    average_fps = sum(fps_list) / len(fps_list)
else:
    average_fps = 0

# Calculate average detections per frame (as a simple accuracy measure)
if frame_count > 0:
    avg_detections_per_frame = total_detected / frame_count
else:
    avg_detections_per_frame = 0


# Display results
print("\n==== YOLO Model Evaluation Summary ====")
print(f"Model: yolo11l.pt")
print(f"Total frames processed: {frame_count}")
print(f"Total persons detected: {total_detected}")
print(f"Average detections per frame: {avg_detections_per_frame:.2f}")
print(f"Average FPS: {average_fps:.2f}")
print("=======================================\n")
print("Video saved as 'output_person_tracked.avi'")
print("Tracking summary saved to 'tracking_summary.csv'")


100%|██████████| 49.0M/49.0M [00:00<00:00, 145MB/s]



0: 384x640 26 persons, 36.3ms
Speed: 2.7ms preprocess, 36.3ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 26 persons, 31.2ms
Speed: 3.2ms preprocess, 31.2ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 26 persons, 26.6ms
Speed: 8.4ms preprocess, 26.6ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 26 persons, 26.1ms
Speed: 3.7ms preprocess, 26.1ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 26 persons, 26.1ms
Speed: 3.3ms preprocess, 26.1ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 26 persons, 26.1ms
Speed: 3.4ms preprocess, 26.1ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 26 persons, 26.1ms
Speed: 3.3ms preprocess, 26.1ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 26 persons, 26.3ms
Speed: 3.3ms preprocess, 26.3ms inference, 2.2ms postprocess per image at

**USING YOLO11x**

In [ ]:
from ultralytics import YOLO
from collections import defaultdict
import numpy as np
import cv2
import csv
import time

# Load YOLO model (Change this when testing other YOLO variants)
model = YOLO("yolo11x.pt")  # You can replace with "yolo11s.pt", "yolo11l.pt", etc.

# Open video
cap = cv2.VideoCapture("/content/video8.mp4")

# Get video properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_input = cap.get(cv2.CAP_PROP_FPS)

# Output video writer
out = cv2.VideoWriter("output_person_tracked.avi",
                      cv2.VideoWriter_fourcc(*'XVID'),
                      fps_input,
                      (frame_width, frame_height))

# For storing tracking trail of each ID
track_history = defaultdict(lambda: [])

# For FPS calculation
prev_time = 0
fps_list = []  # To store FPS values per frame

# For accuracy
total_detected = 0  # Total people detected
frame_count = 0     # Total frames processed

# Create a set to store unique person IDs
unique_ids = set()

# Loop through video
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Perform tracking on the current frame (only for person class: class 0)
    results = model.track(source=frame, persist=True, classes=[0])

    # Only process if there are tracked objects
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        annotated_frame = results[0].plot()

        # Update total detections and frame count
        total_detected += len(track_ids)  # Count of persons detected in this frame
        frame_count += 1

        # Draw tracking trails
        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            unique_ids.add(track_id)
            track = track_history[track_id]
            track.append((float(x), float(y)))
            if len(track) > 30:
                track.pop(0)

            # Draw tracking lines as polylines
            points = np.hstack(track).astype(np.int32).reshape(-1, 1, 2)
            cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 0, 0), thickness=10)

        # Calculate FPS
        cur_time = time.time()
        fps = 1 / (cur_time - prev_time)
        prev_time = cur_time
        fps_list.append(fps)

        # Display FPS on video
        cv2.putText(annotated_frame, "fps: " + str(int(fps)), (10, 70),
                    cv2.FONT_HERSHEY_PLAIN, 3, (255, 0, 255), 3)

        # Save output frame
        out.write(annotated_frame)

# Release everything
cap.release()
out.release()
cv2.destroyAllWindows()

# Save tracking trail summary to CSV
with open("tracking_summary.csv", mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["person_id", "tracking_trail_points", "start_point", "end_point"])
    for person_id, trail in track_history.items():
        if len(trail) >= 2:
            start_point = trail[0]
            end_point = trail[-1]
            writer.writerow([person_id, str(trail), str(start_point), str(end_point)])

# ----------- RESULTS SUMMARY -----------

#number of persons detected
total_detected = len(unique_ids)

# Calculate average FPS
if len(fps_list) > 0:
    average_fps = sum(fps_list) / len(fps_list)
else:
    average_fps = 0

# Calculate average detections per frame (as a simple accuracy measure)
if frame_count > 0:
    avg_detections_per_frame = total_detected / frame_count
else:
    avg_detections_per_frame = 0


# Display results
print("\n==== YOLO Model Evaluation Summary ====")
print(f"Model: yolo11x.pt")
print(f"Total frames processed: {frame_count}")
print(f"Total persons detected: {total_detected}")
print(f"Average detections per frame: {avg_detections_per_frame:.2f}")
print(f"Average FPS: {average_fps:.2f}")
print("=======================================\n")
print("Video saved as 'output_person_tracked.avi'")
print("Tracking summary saved to 'tracking_summary.csv'")


100%|██████████| 109M/109M [00:01<00:00, 110MB/s]



0: 384x640 21 persons, 59.0ms
Speed: 2.5ms preprocess, 59.0ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 21 persons, 38.5ms
Speed: 3.2ms preprocess, 38.5ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 23 persons, 37.7ms
Speed: 3.7ms preprocess, 37.7ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 23 persons, 37.6ms
Speed: 3.6ms preprocess, 37.6ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 23 persons, 37.5ms
Speed: 3.3ms preprocess, 37.5ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 23 persons, 37.6ms
Speed: 3.1ms preprocess, 37.6ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 23 persons, 37.6ms
Speed: 3.9ms preprocess, 37.6ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 23 persons, 37.6ms
Speed: 3.8ms preprocess, 37.6ms inference, 1.9ms postprocess per image at

In [ ]:
#google drive link for chceking output video and trail points csv
https://drive.google.com/drive/folders/1dvDjYi_l9OXSxSLjXvnWPQ-h__eG0ya9?usp=sharing